In [ ]:
import os
import sys
import json
import copy
import re
from datetime import datetime

from openai import OpenAI
import yaml
import numpy as np

sys.path.append("../")
from src.utils.parser import *
from src.utils.feature_utils import theta_to_language, theta_to_state_mask

# Set your OpenAI API key
try:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
except KeyError:
    # read from ~/.openai_api_key
    with open(os.path.expanduser("~/.openai_api_key"), "r") as f:
        api_key = f.read().strip()
    client = OpenAI(api_key=api_key)

%load_ext autoreload
%autoreload 2

In [ ]:
task_instruction = "The task is to move a coffee grasped with the robot's end effector where there is a human user nearby."

# Key state dimensions used to describe the trajectory to the LLM.
# The full state vector has additional dimensions, but these are the ones
# that directly correspond to objects the instruction may refer to.
STATE_DIMENSIONS = [
    ("eef_x", 0),
    ("eef_y", 1),
    ("eef_z", 2),
    ("eef_rot_z", 9),
    ("human_x", 12),
    ("human_y", 13),
    ("laptop_x", 15),
    ("laptop_y", 16),
    ("table_height", 18),
]


def split_trajectory(traj_states):
    """Format a trajectory as a table of key state dimensions for the LLM prompt."""
    if traj_states is None or len(traj_states) == 0:
        return ""
    traj_states = np.asarray(traj_states)
    names = [n for n, _ in STATE_DIMENSIONS]
    header = "Timestep | " + " | ".join(f"{n:>12s}" for n in names)
    sep = "-" * len(header)
    lines = [header, sep]
    for t, state in enumerate(traj_states):
        row = " | ".join(f"{state[idx]:12.3f}" for _, idx in STATE_DIMENSIONS)
        lines.append(f"{t:8d} | {row}")
    return "\n".join(lines)


def is_instruction_ambiguous(
    instruction: str,
    model_text: str = "gpt-4o-mini",
) -> dict:
    """Check whether an instruction is ambiguous.
    
    Returns a dict: {"ambiguous": bool, "explanation": str}.
    """
    messages = [
        {"role": "system", "content": (
            "You are interfacing with a robotics environment in which a robotic arm is sitting on a table. "
            "There is also a laptop on the table and a human standing next to the table. The robotic arm is "
            "learning to manipulate a cup based on some language command (e.g. 'Stay close to the table "
            "surface. Carry the cup upright.'). The user will specify the command that they would like to "
            "teach the robot. However, sometimes the user command is ambiguous, for example it may be "
            "missing the referent (e.g., 'Stay close'), or the desired behavior (e.g., 'Cup'). You will be "
            "asked to assess if the user command is ambiguous or not. Please respond ONLY with JSON: "
            '{"ambiguous":<true|false>,"explanation":...}'
        )},
        {"role": "user", "content": f"Instruction:\n{instruction}"},
    ]
    resp = client.chat.completions.create(
        model=model_text,
        messages=messages,
        temperature=0,
    )
    content = resp.choices[0].message.content.strip()
    if content.startswith("```json"):
        content = content[len("```json"):]
    if content.startswith("```"):
        content = content[3:]
    if content.endswith("```"):
        content = content[:-3]
    return json.loads(content.strip())


def disambiguate_instruction(
    instruction: str,
    traj_states: list,
    model_text: str = "gpt-4o-mini",
    temperature: float = 0.3,
) -> list:
    """Rewrite an ambiguous instruction into one or two clear directives using a trajectory.
    
    Returns a JSON list of 1-2 candidate disambiguated instructions.
    """
    demo_desc = split_trajectory(traj_states)
    system_content = (
        "You are a language assistant interfacing with a robotics environment.\n\n"
        "Environment:\n"
        "A robotic arm is sitting on a table. There is also a laptop on the table and a human standing next "
        "to the table. The robotic arm is learning to manipulate a cup based on some language command "
        "(e.g. 'Stay close to the table surface. Carry the cup upright.'). The user will specify the command "
        "that they would like to teach the robot. Sometimes the user's command is ambiguous, for example it "
        "may be missing the referent (e.g., 'Stay close'), or the desired behavior (e.g., 'Cup').\n\n"
        "Task:\n"
        "Use the trajectory state sequence to disambiguate the command by filling in the missing referent "
        "(e.g., 'Stay close' -> 'Stay close to the table surface' or 'Stay close to the human') and/or the "
        "desired behavior (e.g., 'Cup' -> 'Keep the cup upright' / 'Keep the cup upside down'). Ground "
        "everything in visible elements of the scene: {human, laptop, table, table surface, cup}. Do not "
        "invent objects. Each command should reference a single object.\n\n"
        "Output:\n"
        "Walk through your reasoning based on the input command and the trajectory. Prefer a single, "
        "most-supported interpretation; if two interpretations are equally plausible, output exactly two. "
        "On the FINAL LINE, output ONLY a JSON list of 1-2 strings with the plausible disambiguated "
        "command(s). Examples:\n"
        '["Stay close to the table surface"]\n'
        '["Stay close to the human", "Stay close to the laptop"]'
    )
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": (
            f"Original Instruction:\n{instruction}\n\n"
            f"Split trajectory states:\n{demo_desc}"
        )},
    ]
    resp = client.chat.completions.create(
        model=model_text,
        messages=messages,
        temperature=temperature,
    )
    content = resp.choices[0].message.content.strip()
    last_line = content.splitlines()[-1].strip()
    return json.loads(last_line)


def process_instruction(
    instruction: str,
    traj_states: list,
    model_text: str = "gpt-4o-mini",
    temperature: float = 0.3,
    max_trials: int = 3,
) -> dict:
    """Check ambiguity, then disambiguate if needed, with retries."""
    ambiguity = None
    for trial in range(max_trials):
        try:
            ambiguity = is_instruction_ambiguous(instruction, model_text=model_text)
            break
        except Exception as e:
            print(f"Ambiguity check error (trial {trial + 1}/{max_trials}): {e}")
    if ambiguity is None:
        return None

    clear_instruction = [instruction]
    if ambiguity.get("ambiguous"):
        for trial in range(max_trials):
            try:
                clear_instruction = disambiguate_instruction(
                    instruction,
                    traj_states=traj_states,
                    model_text=model_text,
                    temperature=temperature,
                )
                break
            except Exception as e:
                print(f"Disambiguation error (trial {trial + 1}/{max_trials}): {e}")
        else:
            return None

    return {
        "ambiguous": ambiguity.get("ambiguous", False),
        "explanation": ambiguity.get("explanation", ""),
        "disambiguated_instruction": clear_instruction,
    }

In [ ]:
def get_theta_key(theta):
    """Generate an interpretable key for a given theta vector."""
    return '_'.join(str(x) for x in theta)


# Load configuration files
human_config = "../config/humans/frankarobot_multiple_humans.yaml"
with open(human_config, "r") as stream:
    humans_params_list = yaml.safe_load(stream)

config = "../config/reward_learning/obj20_sg10_persg5/maskedrl_llm_mask.yaml"
with open(config, "r") as stream:
    params = yaml.safe_load(stream)

# Load data split configuration
indices_file = os.path.join(params["irl"]["data_split_config_path"], "split_indices.json")
all_trajs, train_trajs, test_trajs = load_split_data(
    params["env"]["trajset_file"],
    params["env"]["per_SG"],
    params["env"]["train_test_split"],
    indices_file=indices_file,
)
print(f"Total trajectories: {len(all_trajs)} | Train: {len(train_trajs)} | Test: {len(test_trajs)}")

seed = 12345
set_seed(seed)

# Load demo indices for each theta
demo_queries = 10
demo_indices_file = os.path.join(params["irl"]["data_split_config_path"], f"demo_indices_100_{seed}.json")
with open(demo_indices_file, "r") as f:
    saved_demo_indices = json.load(f)
    for key in saved_demo_indices.keys():
        saved_demo_indices[key] = saved_demo_indices[key][:demo_queries]
print(f"Loaded demo indices from {demo_indices_file}")

# Disambiguate Ambiguous Language Instructions

For each human, take the ambiguous instruction variants (`omit_referent` and `omit_expression`) and use an LLM
with the trajectory state sequence to produce disambiguated instructions.

In [ ]:
results_list = copy.deepcopy(humans_params_list["humans"])
resume_path = None
previous_resume_path = None
temperature = 0.3
state_dim = 19

# Load resume data if specified
if resume_path is not None:
    with open(resume_path, "r") as f:
        resume_results_list = json.load(f)
    print(f"Resuming from {resume_path}")

ambiguity_types = ["omit_referent", "omit_expression"]

for human_idx, human_params in enumerate(humans_params_list["humans"]):
    # Skip if already processed (when resuming)
    if (
        resume_path is not None
        and "disambiguated_instructions" in resume_results_list[human_idx]
    ):
        results_list[human_idx] = resume_results_list[human_idx]
        continue

    print(f"Processing human {human_idx}/{len(humans_params_list['humans'])}")

    theta = human_params["preferencer"]["theta"]
    theta_key = get_theta_key(theta)
    gt_state_mask = theta_to_state_mask(theta, state_dim=state_dim).astype(int)
    language_instruction = theta_to_language([theta])[0]

    print(f"Human theta: {theta}")
    print(f"Language instruction: {language_instruction}")

    results_list[human_idx]["language_instruction"] = language_instruction
    results_list[human_idx]["theta_key"] = theta_key
    results_list[human_idx]["gt_state_mask"] = gt_state_mask

    # Skip thetas with no demo indices available
    if theta_key not in saved_demo_indices:
        print(f"No demo indices for theta {theta_key}, skipping")
        continue

    demo_indices = saved_demo_indices[theta_key]
    results_list[human_idx]["demo_indices"] = demo_indices
    results_list[human_idx]["disambiguated_instructions"] = {
        a: {"ambiguous_instruction": None, "per_demo": {}} for a in ambiguity_types
    }

    for ambiguity_type in ambiguity_types:
        ambiguous_instruction = theta_to_language([theta], language_ambiguity=ambiguity_type)[0]
        print(f"  [{ambiguity_type}] Ambiguous instruction: {ambiguous_instruction}")
        results_list[human_idx]["disambiguated_instructions"][ambiguity_type][
            "ambiguous_instruction"
        ] = ambiguous_instruction

        for demo_idx in demo_indices:
            traj_states = all_trajs[demo_idx]
            processed = process_instruction(
                ambiguous_instruction,
                traj_states=traj_states,
                model_text="gpt-4o-mini",
                temperature=temperature,
            )
            if processed is None:
                print(f"    demo {demo_idx}: failed to disambiguate")
                continue

            results_list[human_idx]["disambiguated_instructions"][ambiguity_type]["per_demo"][
                str(demo_idx)
            ] = {
                "ambiguous": processed["ambiguous"],
                "disambiguated_instruction": processed["disambiguated_instruction"],
            }
            print(
                f"    demo {demo_idx}: ambiguous={processed['ambiguous']} -> "
                f"{processed['disambiguated_instruction']}"
            )

    # Save checkpoint every 12 humans
    if human_idx % 12 == 0 and human_idx > 0:
        if previous_resume_path is not None:
            os.remove(previous_resume_path)
            print(f"Removed previous resume file {previous_resume_path}")

        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_path = os.path.join(
            params["irl"]["data_split_config_path"],
            f"language_disambiguated_temp{temperature}_{seed}_{timestamp_str}.json",
        )
        with open(results_path, "w") as f:
            json.dump(results_list, f, indent=4, default=jsonNpEncoder)
        print(f"Saved checkpoint to {results_path}")
        previous_resume_path = results_path

# Save final results
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
results_path = os.path.join(
    params["irl"]["data_split_config_path"],
    f"language_disambiguated_temp{temperature}_{seed}_{timestamp_str}.json",
)
with open(results_path, "w") as f:
    json.dump(results_list, f, indent=4, default=jsonNpEncoder)
print(f"Saved final results to {results_path}")